# 03 Gold Layer
**EAS 587 Phase 3 | Drug Overdose CDC Dataset**

Reads from Silver. Creating real use business aggregates, ML features, and multi-source joined tables.

| Gold Table | Description |
|---|---|
| `gold_state_annual_metrics` | State × year death totals + YoY change |
| `gold_indicator_trends` | National monthly trend per drug category |
| `gold_ml_features` | ML-ready: state × drug_category × year + `high_burden` label |
| `gold_cdc_kff_joined` | CDC overdose deaths joined with KFF OUD rates by state (Task 2) |
| `gold_insight1_oud_vs_deaths` | States ranked by OUD rate vs actual death count |
| `gold_insight2_risk_tier_deaths` | Avg deaths grouped by OUD risk tier |
| `gold_insight3_adolescent_burden` | States with high adolescent OUD but low death count - hidden burden |

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

DB_NAME = "eas587_phase3"

silver_cdc = spark.table(f"{DB_NAME}.silver_cdc_overdose")
silver_kff = spark.table(f"{DB_NAME}.silver_kff_opioid_disorder")
print(f"silver_cdc : {silver_cdc.count():,} rows")
print(f"silver_kff : {silver_kff.count()} rows")

silver_cdc.show(5)

silver_cdc : 23,022 rows
silver_kff : 51 rows
+----------+----------+---------------+-------------+----------+----+-----------+---------------+------------+-------------+---------------------+
|state_code|state_name|      indicator|drug_category|      date|year|death_count|predicted_value|pct_complete|  pct_pending|actual_monthly_deaths|
+----------+----------+---------------+-------------+----------+----+-----------+---------------+------------+-------------+---------------------+
|        AL|   Alabama|Cocaine (T40.5)|      cocaine|2023-04-01|2023|      268.0|          275.0|       100.0|0.23515378717|                 50.0|
|        AL|   Alabama|Cocaine (T40.5)|      cocaine|2023-05-01|2023|      261.0|          268.0|       100.0|0.23558124163|                 31.0|
|        AL|   Alabama|Cocaine (T40.5)|      cocaine|2023-06-01|2023|      271.0|          278.0|       100.0|0.23120208082|                 31.0|
|        AL|   Alabama|Cocaine (T40.5)|      cocaine|2023-07-01|2023|   

## Gold 1 - State Annual Metrics

In [0]:
total_deaths = silver_cdc.filter(F.col("drug_category") == "total_overdose")

state_annual = (
    total_deaths
    .groupBy("state_code", "state_name", "year")
    .agg(
        F.sum("actual_monthly_deaths").alias("annual_deaths"),
        F.avg("actual_monthly_deaths").alias("avg_monthly_deaths"),
        F.max("actual_monthly_deaths").alias("peak_month_deaths"),
        F.stddev("actual_monthly_deaths").alias("stddev_monthly_deaths"),
        F.count("date").alias("months_reported")
    )
    .fillna({"stddev_monthly_deaths": 0.0})
)

state_annual.show(5)

# YoY % change via window lag
w = Window.partitionBy("state_code").orderBy("year")
state_annual = (
    state_annual
    .withColumn("prev_year_deaths", F.lag("annual_deaths", 1).over(w))
    .withColumn("yoy_pct_change",
        F.when(
            F.col("prev_year_deaths").isNotNull() & (F.col("prev_year_deaths") > 0),
            F.round(
                (F.col("annual_deaths") - F.col("prev_year_deaths")) /
                F.col("prev_year_deaths") * 100, 2
            )
        ).otherwise(F.lit(None))
    )
    .drop("prev_year_deaths")
)

print(f"State annual rows: {state_annual.count():,}")
state_annual.orderBy(F.desc("year"), F.desc("annual_deaths")).show(15)

+----------+----------+----+-------------+------------------+-----------------+---------------------+---------------+
|state_code|state_name|year|annual_deaths|avg_monthly_deaths|peak_month_deaths|stddev_monthly_deaths|months_reported|
+----------+----------+----+-------------+------------------+-----------------+---------------------+---------------+
|        AL|   Alabama|2017|       1208.0|100.66666666666667|            158.0|   35.714354978287815|             12|
|        AL|   Alabama|2018|         62.0|              31.0|             40.0|   12.727922061357855|              2|
|        AL|   Alabama|2019|          6.0|               6.0|              6.0|                  0.0|              1|
|        AL|   Alabama|2020|       1981.0|165.08333333333334|            250.0|    69.61381567859452|             12|
|        AL|   Alabama|2021|       3472.0| 289.3333333333333|            349.0|    40.72375525935989|             12|
+----------+----------+----+-------------+--------------

In [0]:
state_annual.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB_NAME}.gold_state_annual_metrics")
print("✅ gold_state_annual_metrics written")

✅ gold_state_annual_metrics written


## Gold 3 - ML Feature Table

In [0]:
# state × drug_category × year aggregated features
ml_base = (
    silver_cdc
    .filter(F.col("year").between(2016, 2023))   # exclude partial boundary years
    .groupBy("state_code", "state_name", "drug_category", "indicator", "year")
    .agg(
        F.sum("death_count").alias("annual_deaths"),
        F.avg("death_count").alias("avg_monthly_deaths"),
        F.max("death_count").alias("max_monthly_deaths"),
        F.stddev("death_count").alias("stddev_deaths"),
        F.count("date").alias("months_reported"),
        F.avg("pct_pending").alias("avg_pct_pending")
    )
    .fillna({"stddev_deaths": 0.0})
)

# YoY % change per state × drug_category
w2 = Window.partitionBy("state_code", "drug_category").orderBy("year")
ml_features = (
    ml_base
    .withColumn("prev_deaths", F.lag("annual_deaths", 1).over(w2))
    .withColumn("yoy_pct_change",
        F.when(
            F.col("prev_deaths").isNotNull() & (F.col("prev_deaths") > 0),
            F.round(
                (F.col("annual_deaths") - F.col("prev_deaths")) / F.col("prev_deaths") * 100, 2
            )
        ).otherwise(F.lit(0.0))
    )
    .drop("prev_deaths")
    # Binary label: 1 = top 50% death count for that drug_category + year across all states
    .withColumn("death_pct_rank",
        F.percent_rank().over(
            Window.partitionBy("drug_category", "year").orderBy("annual_deaths")
        )
    )
    .withColumn("high_burden", F.when(F.col("death_pct_rank") >= 0.5, 1).otherwise(0))
    .drop("death_pct_rank")
)

print(f"ML feature rows: {ml_features.count():,}")
ml_features.show(10)

ML feature rows: 2,529
+----------+-------------+-------------+---------------+----+-------------+------------------+------------------+------------------+---------------+--------------------+--------------+-----------+
|state_code|   state_name|drug_category|      indicator|year|annual_deaths|avg_monthly_deaths|max_monthly_deaths|     stddev_deaths|months_reported|     avg_pct_pending|yoy_pct_change|high_burden|
+----------+-------------+-------------+---------------+----+-------------+------------------+------------------+------------------+---------------+--------------------+--------------+-----------+
|        IA|         Iowa|      cocaine|Cocaine (T40.5)|2016|         47.0|15.666666666666666|              16.0|0.5773502691896258|              3|0.006873684283333334|           0.0|          0|
|        OK|     Oklahoma|      cocaine|Cocaine (T40.5)|2016|        121.0|             30.25|              32.0|1.2583057392117916|              4|     0.0151052886125|           0.0|     

In [0]:
ml_features.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB_NAME}.gold_ml_features")
print("✅ gold_ml_features written")

✅ gold_ml_features written


---
## Gold 4 - CDC × KFF Join

**Join key:** `state_name`  
**Join type:** Left join from state annual CDC data onto KFF OUD rates  

This enriches each state-year record with the OUD prevalence rate for that state,
enabling analysis of whether states with higher opioid use disorder rates also
show higher overdose death counts.

In [0]:
# state_annual already built in Gold 1 - reload from table
gold_state_annual = spark.table(f"{DB_NAME}.gold_state_annual_metrics")

##joining the additional datset
gold_joined = (
    gold_state_annual
    .join(silver_kff, on="state_name", how="left")
)



print(f"Joined rows: {gold_joined.count():,}")
# Check join quality - how many states matched KFF data
matched   = gold_joined.filter(F.col("adult_oud_pct").isNotNull()).select("state_name").distinct().count()
unmatched = gold_joined.filter(F.col("adult_oud_pct").isNull()).select("state_name").distinct().count()
print(f"States matched to KFF data  : {matched}")
print(f"States unmatched (no KFF row): {unmatched}")

gold_joined.select(
    "state_name", "year", "annual_deaths",
    "adult_oud_pct", "adolescent_oud_pct", "oud_risk_tier"
).orderBy(F.desc("year"), F.desc("annual_deaths")).show(15, truncate=False)

Joined rows: 407
States matched to KFF data  : 51
States unmatched (no KFF row): 3
+------------+----+-------------+-------------+------------------+-------------+
|state_name  |year|annual_deaths|adult_oud_pct|adolescent_oud_pct|oud_risk_tier|
+------------+----+-------------+-------------+------------------+-------------+
|Arizona     |2025|2141.0       |2.4          |1.3               |medium       |
|New Mexico  |2025|116.0        |3.0          |1.0               |high         |
|Colorado    |2025|61.0         |2.1          |1.0               |low          |
|Hawaii      |2025|49.0         |2.1          |0.7               |low          |
|Utah        |2025|24.0         |2.0          |1.1               |low          |
|South Dakota|2025|13.0         |2.1          |1.3               |low          |
|North Dakota|2025|7.0          |1.8          |1.3               |low          |
|Kansas      |2025|5.0          |2.2          |1.5               |medium       |
|Montana     |2025|5.0    

In [0]:
gold_joined.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB_NAME}.gold_cdc_kff_joined")
print("✅ gold_cdc_kff_joined written")

✅ gold_cdc_kff_joined written


---
## Gold 5 - Task 2 Insights (3 Required)

**Insight 1:** States with highest OUD rates vs actual overdose deaths - do they align?  
**Insight 2:** Average annual overdose deaths grouped by OUD risk tier (high/medium/low)  
**Insight 3:** States with high adolescent OUD rate but relatively lower total deaths - hidden burden

In [0]:
gold_kff = spark.table(f"{DB_NAME}.gold_cdc_kff_joined")

#INSIGHT 1: OUD rate vs overdose deaths - 2022/2023 most recent year 

insight_1 = (
    gold_kff
    .filter(F.col("year") == 2022)
    .filter(F.col("adult_oud_pct").isNotNull())
    .select(
        "state_name", "annual_deaths", "adult_oud_pct",
        "adolescent_oud_pct", "combined_oud_pct",
        "oud_risk_tier"
    )
    .orderBy(F.desc("adult_oud_pct"))
)
print("INSIGHT 1: OUD Rate vs Overdose Deaths by State (2022)")
insight_1.show(15, truncate=False)

INSIGHT 1: OUD Rate vs Overdose Deaths by State (2022)
+--------------+-------------+-------------+------------------+----------------+-------------+
|state_name    |annual_deaths|adult_oud_pct|adolescent_oud_pct|combined_oud_pct|oud_risk_tier|
+--------------+-------------+-------------+------------------+----------------+-------------+
|Louisiana     |2628.0       |4.0          |1.2               |2.6             |high         |
|West Virginia |105.0        |3.7          |0.9               |2.3             |high         |
|Mississippi   |505.0        |3.6          |1.0               |2.3             |high         |
|Kentucky      |481.0        |3.3          |1.4               |2.35            |high         |
|New Mexico    |1129.0       |3.0          |1.0               |2.0             |high         |
|Oklahoma      |3149.0       |2.8          |1.4               |2.1             |medium       |
|Alabama       |2569.0       |2.7          |1.0               |1.85            |medium    

In [0]:
# INSIGHT 2: Avg annual deaths by OUD risk tier 
# Does being in a 'high' OUD risk tier translate to significantly more deaths -- NO from this data
insight_2 = (
    gold_kff
    .filter(F.col("oud_risk_tier").isNotNull())
    .groupBy("oud_risk_tier")
    .agg(
        F.round(F.avg("annual_deaths"), 0).alias("avg_annual_deaths"),
        F.round(F.avg("adult_oud_pct"), 3).alias("avg_adult_oud_pct"),
        F.count("state_name").alias("state_year_records"),
        F.countDistinct("state_name").alias("distinct_states")
    )
    .orderBy(F.desc("avg_adult_oud_pct"))
)
print("INSIGHT 2: Avg Annual Overdose Deaths by OUD Risk Tier")
insight_2.show(truncate=False)

INSIGHT 2: Avg Annual Overdose Deaths by OUD Risk Tier
+-------------+-----------------+-----------------+------------------+---------------+
|oud_risk_tier|avg_annual_deaths|avg_adult_oud_pct|state_year_records|distinct_states|
+-------------+-----------------+-----------------+------------------+---------------+
|high         |1756.0           |3.468            |34                |5              |
|medium       |1671.0           |2.338            |177               |22             |
|low          |2232.0           |1.985            |179               |24             |
+-------------+-----------------+-----------------+------------------+---------------+



In [0]:
# ── INSIGHT 3: High adolescent OUD but lower death count - hidden burden(tells about adolescent deaths in 2022)
# States where adolescent OUD is disproportionately high relative to total death count.
# These are states where the crisis may be under-counted in mortality statistics.
insight_3 = (
    gold_kff
    .filter(F.col("year") == 2022)
    .filter(F.col("adolescent_oud_pct").isNotNull())
    # Rank states: adolescent OUD rank (desc) minus death rank (desc)
    .withColumn("adolescent_oud_rank",
        F.rank().over(Window.orderBy(F.desc("adolescent_oud_pct")))
    )
    .withColumn("death_count_rank",
        F.rank().over(Window.orderBy(F.desc("annual_deaths")))
    )
    # Large positive value = high OUD rank but low death rank = hidden burden
    .withColumn("hidden_burden_score", F.col("death_count_rank") - F.col("adolescent_oud_rank"))
    .select(
        "state_name", "adolescent_oud_pct", "adult_oud_pct",
        "annual_deaths", "adolescent_oud_rank", "death_count_rank",
        "hidden_burden_score"
    )
    .orderBy(F.desc("hidden_burden_score"))
)
print("INSIGHT 3: States with High Adolescent OUD but Relatively Low Death Count (2022)")
print("(Positive hidden_burden_score = OUD burden exceeds what deaths alone suggest)")
insight_3.show(15, truncate=False)

INSIGHT 3: States with High Adolescent OUD but Relatively Low Death Count (2022)
(Positive hidden_burden_score = OUD burden exceeds what deaths alone suggest)
+--------------------+------------------+-------------+-------------+-------------------+----------------+-------------------+
|state_name          |adolescent_oud_pct|adult_oud_pct|annual_deaths|adolescent_oud_rank|death_count_rank|hidden_burden_score|
+--------------------+------------------+-------------+-------------+-------------------+----------------+-------------------+
|Nebraska            |1.8               |2.1          |46.0         |1                  |47              |46                 |
|North Dakota        |1.3               |1.8          |122.0        |11                 |44              |33                 |
|South Dakota        |1.3               |2.1          |159.0        |11                 |42              |31                 |
|Wyoming             |1.3               |2.3          |188.0        |11        

In [0]:
insight_1.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB_NAME}.gold_insight1_oud_vs_deaths")
insight_2.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB_NAME}.gold_insight2_risk_tier_deaths")
insight_3.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB_NAME}.gold_insight3_adolescent_burden")
print("✅ All 3 insight tables written")

✅ All 3 insight tables written


## Gold Layer Summary

| Table | Description |
|---|---|
| `gold_state_annual_metrics` | State × year totals + YoY change |
| `gold_indicator_trends` | National monthly trend per drug category |
| `gold_ml_features` | ML-ready features + `high_burden` label |
| `gold_cdc_kff_joined` | CDC deaths joined with KFF OUD rates on `state_name` |
| `gold_insight1_oud_vs_deaths` | OUD rate vs overdose deaths by state (2022) |
| `gold_insight2_risk_tier_deaths` | Avg deaths by OUD risk tier |
| `gold_insight3_adolescent_burden` | States with high adolescent OUD but low death count |

